# 5. Build Annotation-Derived Baseline c_l for Selected Loci

This workbook aligns the selected loci across Phase 1 GTEx summary stats, LD, and AlphaGenome outputs, then estimates a conservative per-locus baseline annotation scale.

Per locus:

- `c_rms_l = RMS(beta_hat_std)` because `a` is RMS-normalized within locus
- `c_cap_l = q95(|beta_hat_std|) / max(|a|)` as a safety cap
- `baseline_c_l = min(c_rms_l, c_cap_l)`

There is no pooled or median `c` in this notebook. The goal here is only to report the baseline per-locus scale.

In [ ]:
from pathlib import Path
from statistics import NormalDist
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

In [ ]:
NORMAL = NormalDist()
BOUNDARY_EPS = 1e-4
WINSOR_LIMIT = 2.5
VAR_Y_MEDIAN_RANGE = (0.8, 1.25)
VAR_Y_MAX_IQR = 0.5


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / 'utils').is_dir() and (candidate / 'vignettes').is_dir() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('Could not locate eQTL_annotations_for_susine project root.')


def resolve_selection_path(project_root):
    selection_dir = project_root / 'output' / 'prelim' / 'phase1_metrics_screening_review'
    candidates = sorted(selection_dir.glob('representative_gene_sample_n*_annotation_selection.csv'))
    if not candidates:
        raise FileNotFoundError(f'Could not find selection CSV under {selection_dir}')
    return candidates[-1]


def merge_manifest_selection(manifest_df, selection_df):
    merged = manifest_df.merge(selection_df, on='locus_id', how='inner', suffixes=('_manifest', '_selection'))
    for field in ['gene_name', 'gene_id', 'gtex_tissue', 'gtex_chrom', 'priority', 'notes']:
        m_col, s_col = f'{field}_manifest', f'{field}_selection'
        if m_col in merged.columns and s_col in merged.columns:
            merged[field] = merged[s_col].combine_first(merged[m_col])
        elif s_col in merged.columns:
            merged[field] = merged[s_col]
        elif m_col in merged.columns:
            merged[field] = merged[m_col]
    return merged


def build_dense_ld(ld_long_df, p):
    R = np.zeros((p, p), dtype=float)
    np.fill_diagonal(R, 1.0)
    i = ld_long_df['snp_index_1'].to_numpy(dtype=int)
    j = ld_long_df['snp_index_2'].to_numpy(dtype=int)
    r = ld_long_df['r'].to_numpy(dtype=float)
    R[i, j] = r
    R[j, i] = r
    return R


def rms(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.sqrt(np.mean(x**2))) if x.size else float('nan')


def iqr(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return float(np.percentile(x, 75) - np.percentile(x, 25)) if x.size else float('nan')


def safe_corr(a, b, method='pearson'):
    paired = pd.DataFrame({'a': a, 'b': b}).dropna()
    return float(paired['a'].corr(paired['b'], method=method)) if len(paired) >= 2 else float('nan')


def transform_annotation(q):
    q = np.asarray(q, dtype=float)
    q_star = np.clip(q, -1 + BOUNDARY_EPS, 1 - BOUNDARY_EPS)
    a_raw = np.array([NORMAL.inv_cdf((x + 1) / 2) for x in q_star], dtype=float)
    a_clip = np.clip(a_raw, -WINSOR_LIMIT, WINSOR_LIMIT)
    a_rms = rms(a_clip)
    a = np.zeros_like(a_clip) if (not np.isfinite(a_rms) or a_rms == 0) else a_clip / a_rms
    return pd.DataFrame({'q_star': q_star, 'a_raw': a_raw, 'a_clip': a_clip, 'a': a, 'a_rms': a_rms})


def plot_overlap(df, output_path, ncols=5):
    loci = list(df['locus_id'].dropna().unique())
    nrows = math.ceil(len(loci) / ncols)
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(3.4 * ncols, 3.0 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, locus_id in zip(axes, loci):
        g = df[df['locus_id'] == locus_id]
        beta_vals = g['beta_hat_std'].to_numpy(dtype=float)
        mu0_vals = g['baseline_mu0_l'].to_numpy(dtype=float)
        combined = np.concatenate([beta_vals[np.isfinite(beta_vals)], mu0_vals[np.isfinite(mu0_vals)]])
        bins = np.histogram_bin_edges(combined, bins=40) if combined.size else 40
        ax.hist(beta_vals, bins=bins, alpha=0.55, color='#2f6c8f', label='beta_hat_std')
        ax.hist(mu0_vals, bins=bins, alpha=0.45, color='#b85c38', label='baseline mu0 = c_l*a')
        ax.set_title(str(locus_id))
        ax.set_xlabel('Value')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
    for ax in axes[len(loci):]:
        ax.axis('off')
    fig.suptitle('Overlapping histograms: baseline mu0 vs beta_hat_std')
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def plot_var_y(df, output_path):
    plot_df = df.melt(id_vars=['locus_id'], value_vars=['var_y_hat_from_slope', 'var_y_hat_from_se'], var_name='estimator', value_name='var_y_hat')
    plot_df = plot_df[np.isfinite(plot_df['var_y_hat'])]
    estimators = list(plot_df['estimator'].unique())
    loci = list(plot_df['locus_id'].unique())
    fig, axes = plt.subplots(nrows=len(estimators), ncols=len(loci), figsize=(2.6 * len(loci), 2.4 * len(estimators)), squeeze=False)
    for row_idx, estimator in enumerate(estimators):
        for col_idx, locus_id in enumerate(loci):
            ax = axes[row_idx][col_idx]
            vals = plot_df.loc[(plot_df['estimator'] == estimator) & (plot_df['locus_id'] == locus_id), 'var_y_hat'].to_numpy(dtype=float)
            vals = vals[np.isfinite(vals)]
            ax.hist(vals, bins=35, color='#4c9f70', edgecolor='white')
            if row_idx == 0:
                ax.set_title(str(locus_id))
            if col_idx == 0:
                ax.set_ylabel(estimator)
            ax.set_xlabel('var_y_hat')
    fig.suptitle('var_y proxy diagnostics')
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def plot_baseline_c(summary_df, output_path):
    frame = summary_df.sort_values('baseline_c_l', ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = np.where(frame['capped'], '#b85c38', '#2f6c8f')
    ax.barh(frame['locus_id'], frame['baseline_c_l'], color=colors)
    ax.set_title('Per-locus baseline c_l')
    ax.set_xlabel('baseline_c_l')
    ax.set_ylabel('Locus')
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

In [ ]:
project_root = find_project_root()
manifest_path = project_root / 'config' / 'loci_manifest_sample_100_per_chrom.csv'
selection_path = resolve_selection_path(project_root)
selection_name = selection_path.stem
output_dir = project_root / 'output'
ld_dir = output_dir / 'ld'
annotation_dir = output_dir / 'annotation' / 'alphagenome'
review_dir = output_dir / 'susine_mu0' / selection_name
review_dir.mkdir(parents=True, exist_ok=True)
per_locus_annotation_dir = review_dir / 'per_locus_annotations'
per_locus_annotation_dir.mkdir(parents=True, exist_ok=True)

manifest_df = pd.read_csv(manifest_path)
selection_df = pd.read_csv(selection_path)
selection_df = selection_df[selection_df['annotate'].astype(bool)].copy()
selected_loci_df = merge_manifest_selection(manifest_df, selection_df).drop_duplicates('locus_id')
selected_loci_df = selected_loci_df.sort_values(['priority', 'locus_id'] if 'priority' in selected_loci_df.columns else ['locus_id']).reset_index(drop=True)

variant_rows, summary_rows, beta_rows, corr_rows, errors = [], [], [], [], []

for _, row in selected_loci_df.iterrows():
    locus_id, gene_name = str(row['locus_id']), str(row['gene_name'])
    locus_dir = ld_dir / locus_id
    master_path = locus_dir / f'{gene_name}_phase1_master_variants.csv'
    order_path = locus_dir / f'{gene_name}_LD_variant_order.tsv'
    ld_long_path = locus_dir / f'{gene_name}_phase1_LD_R_long.parquet'
    annotation_path = annotation_dir / locus_id / f'{gene_name}_alphagenome_filtered_scores.parquet'
    missing = [str(p) for p in [master_path, order_path, ld_long_path, annotation_path] if not p.exists()]
    if missing:
        errors.append({'locus_id': locus_id, 'gene_name': gene_name, 'status': 'missing_inputs', 'error_message': ' | '.join(missing)})
        continue

    try:
        master_df = pd.read_csv(master_path)
        master_df = master_df.loc[master_df['ld_included'], ['variant_id', 'z_score', 'slope', 'slope_se', 'sample_size', 'af']].copy()
        order_df = pd.read_csv(order_path, sep='\t').sort_values('index').reset_index(drop=True)
        annotation_df = pq.read_table(annotation_path).to_pandas()
        annotation_df = annotation_df.groupby('source_variant_id', as_index=False).agg(
            raw_score=('raw_score', 'mean'),
            quantile_score=('quantile_score', 'mean'),
            source_scoring_mode=('source_scoring_mode', lambda s: s.dropna().iloc[0] if len(s.dropna()) else 'missing'),
        )
        aligned = order_df.merge(master_df, left_on='id', right_on='variant_id', how='left')
        aligned = aligned.merge(annotation_df, left_on='id', right_on='source_variant_id', how='left').sort_values('index').reset_index(drop=True)
        aligned['locus_id'] = locus_id
        aligned['gene_name'] = gene_name
        aligned['variant_id'] = aligned['id']
        aligned['annotation_missing'] = aligned['quantile_score'].isna()
        aligned['raw_score'] = aligned['raw_score'].fillna(0.0)
        aligned['quantile_score'] = aligned['quantile_score'].fillna(0.0)
        aligned['source_scoring_mode'] = aligned['source_scoring_mode'].fillna('missing')
        if aligned['z_score'].isna().any() or aligned['slope'].isna().any() or aligned['slope_se'].isna().any():
            raise ValueError('Missing z/slope/slope_se after alignment.')
        if not np.array_equal(aligned['index'].to_numpy(), np.arange(len(aligned))):
            raise ValueError('LD order index is not contiguous from 0 to p-1.')

        aligned = pd.concat([aligned, transform_annotation(aligned['quantile_score'])], axis=1)
        aligned['adj'] = (aligned['sample_size'] - 1) / (aligned['z_score'] ** 2 + aligned['sample_size'] - 2)
        aligned['beta_hat_std'] = np.sqrt(aligned['adj']) * aligned['z_score'] / np.sqrt(aligned['sample_size'] - 1)
        aligned['beta_hat_slope'] = aligned['slope'] / np.sqrt(aligned['adj'])
        aligned['var_x_proxy'] = 2 * aligned['af'] * (1 - aligned['af'])
        aligned['r2_proxy'] = aligned['z_score'] ** 2 / (aligned['z_score'] ** 2 + aligned['sample_size'] - 2)
        aligned['var_y_hat_from_slope'] = np.where(aligned['r2_proxy'] > 0, aligned['slope'] ** 2 * aligned['var_x_proxy'] / aligned['r2_proxy'], np.nan)
        aligned['var_y_hat_from_se'] = np.where(aligned['r2_proxy'] < 1, aligned['slope_se'] ** 2 * (aligned['sample_size'] - 1) * aligned['var_x_proxy'] / (1 - aligned['r2_proxy']), np.nan)

        q95_abs_beta_hat_std = float(np.nanquantile(np.abs(aligned['beta_hat_std']), 0.95))
        max_abs_a = float(np.nanmax(np.abs(aligned['a']))) if len(aligned) else float('nan')
        c_rms_l = rms(aligned['beta_hat_std'])
        c_cap_l = (q95_abs_beta_hat_std / max_abs_a) if (np.isfinite(max_abs_a) and max_abs_a > 0) else 0.0
        baseline_c_l = min(c_rms_l, c_cap_l) if np.isfinite(c_rms_l) and np.isfinite(c_cap_l) else float('nan')
        aligned['baseline_mu0_l'] = baseline_c_l * aligned['a']

        locus_annotation_path = per_locus_annotation_dir / f'{gene_name}_mu0_variant_annotations.csv'
        aligned_export = aligned[[
            'locus_id', 'gene_name', 'variant_id', 'index', 'annotation_missing',
            'source_scoring_mode', 'raw_score', 'quantile_score', 'q_star',
            'a_raw', 'a_clip', 'a', 'beta_hat_std', 'beta_hat_slope',
            'baseline_mu0_l', 'var_y_hat_from_slope', 'var_y_hat_from_se'
        ]].rename(columns={
            'index': 'ld_matrix_index',
            'a': 'annotation_a',
            'baseline_mu0_l': 'mu0',
        }).copy()
        aligned_export['baseline_c_l'] = baseline_c_l
        aligned_export.to_csv(locus_annotation_path, index=False)

        summary_rows.append({
            'locus_id': locus_id,
            'gene_name': gene_name,
            'n_variants': len(aligned),
            'annotation_missing_count': int(aligned['annotation_missing'].sum()),
            'var_y_median_proxy': float(np.nanmedian(aligned['var_y_hat_from_slope'])),
            'var_y_iqr_proxy': iqr(aligned['var_y_hat_from_slope']),
            'var_y_median_proxy_se': float(np.nanmedian(aligned['var_y_hat_from_se'])),
            'var_y_iqr_proxy_se': iqr(aligned['var_y_hat_from_se']),
            'original_scale_plausible': bool(
                VAR_Y_MEDIAN_RANGE[0] <= float(np.nanmedian(aligned['var_y_hat_from_slope'])) <= VAR_Y_MEDIAN_RANGE[1]
                and VAR_Y_MEDIAN_RANGE[0] <= float(np.nanmedian(aligned['var_y_hat_from_se'])) <= VAR_Y_MEDIAN_RANGE[1]
                and iqr(aligned['var_y_hat_from_slope']) <= VAR_Y_MAX_IQR
                and iqr(aligned['var_y_hat_from_se']) <= VAR_Y_MAX_IQR
            ),
            'scale_route_recommendation': 'standardized path only',
            'q95_abs_beta_hat_std': q95_abs_beta_hat_std,
            'max_abs_a': max_abs_a,
            'c_rms_l': c_rms_l,
            'c_cap_l': c_cap_l,
            'baseline_c_l': baseline_c_l,
            'capped': bool(np.isfinite(c_rms_l) and np.isfinite(c_cap_l) and c_cap_l < c_rms_l),
            'final_annotation_csv': str(locus_annotation_path),
        })
        beta_rows.append({
            'locus_id': locus_id,
            'gene_name': gene_name,
            'beta_hat_primary_path': 'z_standardized',
            'beta_hat_sensitivity_path': 'slope_adjusted',
            'corr_beta_std_beta_slope': safe_corr(aligned['beta_hat_std'], aligned['beta_hat_slope'], 'pearson'),
            'spearman_beta_std_beta_slope': safe_corr(aligned['beta_hat_std'], aligned['beta_hat_slope'], 'spearman'),
            'mean_abs_beta_diff': float(np.nanmean(np.abs(aligned['beta_hat_std'] - aligned['beta_hat_slope']))),
        })
        corr_rows.append({
            'locus_id': locus_id,
            'gene_name': gene_name,
            'corr_a_beta_std': safe_corr(aligned['a'], aligned['beta_hat_std'], 'pearson'),
            'spearman_a_beta_std': safe_corr(aligned['a'], aligned['beta_hat_std'], 'spearman'),
            'corr_quantile_beta_std': safe_corr(aligned['quantile_score'], aligned['beta_hat_std'], 'pearson'),
            'spearman_quantile_beta_std': safe_corr(aligned['quantile_score'], aligned['beta_hat_std'], 'spearman'),
        })
        variant_rows.append(aligned[['locus_id', 'gene_name', 'variant_id', 'beta_hat_std', 'beta_hat_slope', 'a', 'baseline_mu0_l', 'var_y_hat_from_slope', 'var_y_hat_from_se']].copy())
    except Exception as exc:
        errors.append({'locus_id': locus_id, 'gene_name': gene_name, 'status': 'failed', 'error_message': str(exc)})

mu0_variant_df = pd.concat(variant_rows, ignore_index=True) if variant_rows else pd.DataFrame()
mu0_locus_summary_df = pd.DataFrame(summary_rows).sort_values('locus_id').reset_index(drop=True)
beta_path_summary_df = pd.DataFrame(beta_rows).sort_values('locus_id').reset_index(drop=True)
annotation_correlation_df = pd.DataFrame(corr_rows).sort_values('locus_id').reset_index(drop=True)
alignment_errors_df = pd.DataFrame(errors)

mu0_locus_summary_df.to_csv(review_dir / 'mu0_locus_summary.csv', index=False)
beta_path_summary_df.to_csv(review_dir / 'mu0_beta_path_summary.csv', index=False)
annotation_correlation_df.to_csv(review_dir / 'mu0_annotation_correlation_per_locus.csv', index=False)

plot_overlap(mu0_variant_df, review_dir / 'baseline_annotation_vs_beta_overlap.png')
plot_var_y(mu0_variant_df, review_dir / 'var_y_hat_diagnostics.png')
plot_baseline_c(mu0_locus_summary_df, review_dir / 'per_locus_baseline_c.png')

results = {
    'project_root': project_root,
    'selection_path': selection_path,
    'review_dir': review_dir,
    'selected_loci_df': selected_loci_df,
    'per_locus_annotation_dir': per_locus_annotation_dir,
    'mu0_locus_summary_df': mu0_locus_summary_df,
    'beta_path_summary_df': beta_path_summary_df,
    'annotation_correlation_df': annotation_correlation_df,
    'alignment_errors_df': alignment_errors_df,
}
review_dir

## Selected Loci

In [ ]:
results['selected_loci_df'][['locus_id', 'gene_name', 'gtex_tissue', 'gtex_chrom', 'priority', 'notes']]

## Per-Locus Baseline c_l

In [ ]:
results['mu0_locus_summary_df'][[
    'locus_id', 'gene_name', 'q95_abs_beta_hat_std', 'max_abs_a',
    'c_rms_l', 'c_cap_l', 'baseline_c_l', 'capped'
]]

## var_y by Locus

In [ ]:
results['mu0_locus_summary_df'][[
    'locus_id', 'gene_name', 'var_y_median_proxy', 'var_y_iqr_proxy',
    'var_y_median_proxy_se', 'var_y_iqr_proxy_se', 'original_scale_plausible'
]]

## beta_hat Path Comparison

In [ ]:
results['beta_path_summary_df']

## Annotation Correlation by Locus

In [ ]:
results['annotation_correlation_df']

## Alignment Errors

In [ ]:
results['alignment_errors_df']

## Output Files

This notebook now writes per-locus final annotation CSVs plus three review CSVs and three plots:

- `per_locus_annotations/<GENE>_mu0_variant_annotations.csv`
- `mu0_locus_summary.csv`
- `mu0_beta_path_summary.csv`
- `mu0_annotation_correlation_per_locus.csv`
- `baseline_annotation_vs_beta_overlap.png`
- `var_y_hat_diagnostics.png`
- `per_locus_baseline_c.png`

In [ ]:
sorted(path.name for path in review_dir.iterdir())